    Node change: Did an object appear or disappear?

    Relation change: Did a relation start or end?

    Relation duration: How long has a relation remained active?

    Node identity : Which object appeared or disappeared?

    Relation identity: Which action or relation changed?

    Relation start: At which observed frame did the current relation begin?

    Change count: How many node or relation changes occurred?

In [1]:
import os
import glob
import json
import re
from collections import defaultdict

def parse_json_sequence(input_dir):
    """Groups JSON frames by Video ID and sorts them chronologically."""
    video_timelines = defaultdict(list)
    file_paths = glob.glob(os.path.join(input_dir, "*.json"))
    
    for file_path in file_paths:
        filename = os.path.basename(file_path)
        match = re.match(r"VID(\d+)_(\d+)\.json", filename)
        if not match: continue
            
        vid_id = f"VID{match.group(1)}"
        frame_num = int(match.group(2))
        
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        triplets = data.get("info", {}).get("triplet", [])
        
        frame_state = {}
        for t in triplets:
            parts = t.split(',')
            if len(parts) == 3:
                inst, verb, target = [p.strip() for p in parts]
                frame_state[inst] = (verb, target)
                
        video_timelines[vid_id].append({"frame": frame_num, "state": frame_state})
        
    for vid_id in video_timelines:
        video_timelines[vid_id].sort(key=lambda x: x["frame"])
        
    return video_timelines

def extract_action_spans(video_timeline):
    """Groups consecutive identical actions into spans (start, end, duration)."""
    spans_by_inst = defaultdict(list)
    active_spans = {}
    
    for data in video_timeline:
        frame = data["frame"]
        current_state = data["state"]
        
        for inst, (verb, target) in current_state.items():
            if inst in active_spans:
                if active_spans[inst]["verb"] == verb and active_spans[inst]["target"] == target:
                    active_spans[inst]["end_frame"] = frame
                    active_spans[inst]["duration"] += 1
                else:
                    spans_by_inst[inst].append(active_spans[inst])
                    active_spans[inst] = {"start_frame": frame, "end_frame": frame, "duration": 1, "verb": verb, "target": target}
            else:
                active_spans[inst] = {"start_frame": frame, "end_frame": frame, "duration": 1, "verb": verb, "target": target}
                
        ended_insts = [inst for inst in active_spans if inst not in current_state]
        for inst in ended_insts:
            spans_by_inst[inst].append(active_spans[inst])
            del active_spans[inst]
            
    for inst, span in active_spans.items():
        spans_by_inst[inst].append(span)
        
    return spans_by_inst

def find_span_at(spans, frame):
    """Returns the action span of an instrument that is active at `frame`."""
    for span in spans:
        if span["start_frame"] <= frame <= span["end_frame"]:
            return span
    return None

# def generate_single_word_qa(video_timelines):
#     """Generates Single-Word Answer Temporal QA."""
#     all_qa_pairs = {}

#     # Define the values we want to exclude globally for easy maintenance
#     exclude_list = ["null_verb", "null_target", "null_instrument", "none", ""]
    
#     for vid_id, timeline in video_timelines.items():
#         spans = extract_action_spans(timeline)
#         qa_pairs = []
#         frame_lookup = { data["frame"]: data["state"] for data in timeline }
        
#         for inst, actions in spans.items():
#             # Exclude if the instrument itself is null/empty
#             if inst in exclude_list:
#                 continue
                
#             for i, action in enumerate(actions):
#                 verb = action["verb"]
#                 target = action["target"]
#                 start = action["start_frame"]
#                 end = action["end_frame"]
#                 dur = action["duration"]

#                 # 1. ADD THIS LINE: Skip the main action if verb or target is null/empty
#                 if verb in exclude_list or target in exclude_list:
#                     continue

#                 # 1. Number/Time Templates
#                 qa_pairs.append((f"At what exact frame did the {inst} begin {verb}ing the {target}?", f"{start}",'0'))
#                 qa_pairs.append((f"At what exact frame did the {inst} stop {verb}ing the {target}?", f"{end}",'0'))
#                 qa_pairs.append((f"How many consecutive frames did the {inst} spend {verb}ing the {target} starting at frame {start}?", f"{dur}",'0'))

#                 # 2. Sequence Templates (Before / After)
#                 if i + 1 < len(actions):
#                     next_action = actions[i+1]
#                     next_verb = next_action["verb"]
#                     next_target = next_action["target"]

#                     # 2. ADD THIS LINE: Only ask 'next' questions if the next triplet is also valid
#                     if next_verb not in exclude_list and next_target not in exclude_list:
#                         qa_pairs.append((f"What action did the {inst} perform immediately after {verb}ing the {target}?", f"{next_verb}",'1'))
#                         qa_pairs.append((f"What tissue did the {inst} interact with immediately after {verb}ing the {target}?", f"{next_target}",'1'))
                    
#                     # qa_pairs.append((f"What action did the {inst} perform immediately after {verb}ing the {target}?", f"{next_verb}"))
#                     # qa_pairs.append((f"What tissue did the {inst} interact with immediately after {verb}ing the {target}?", f"{next_target}"))

#                 # 3. Concurrency Templates
#                 frame_state = frame_lookup[start]
#                 for other_inst, (other_verb, other_target) in frame_state.items():
#                     # 3. ADD THIS LINE: Skip concurrent instruments if their action/target is null
#                     if other_inst == inst or other_verb in exclude_list or other_target in exclude_list:
#                         continue
#                     # if other_inst != inst:
#                     qa_pairs.append((f"What action was the {other_inst} performing while the {inst} was {verb}ing the {target} at frame {start}?", f"{other_verb}",'2'))
#                     qa_pairs.append((f"What tissue was the {other_inst} {other_verb}ing while the {inst} was {verb}ing the {target} at frame {start}?", f"{other_target}",'2'))
#                     qa_pairs.append((f"Which instrument was {other_verb}ing the {other_target} while the {inst} was {verb}ing the {target} at frame {start}?", f"{other_inst}",'2'))

#         all_qa_pairs[vid_id] = qa_pairs
        
#     return all_qa_pairs

def generate_single_word_qa(video_timelines):
    """Generates Single-Word Answer Temporal QA with action frame ranges.

    Each QA entry is a tuple:
        (question, answer, question_type,
         anchor_start, anchor_end, related_start, related_end)
    anchor_* = frames of the action the QA was generated from (the action the
    answer belongs to), related_* = frames of the other action referenced by
    the question. Frame values are plain integers (numeric only).
    """
    all_qa_pairs = {}
    
    # Global exclude list for easy maintenance
    exclude_list = ["null_verb", "null_target", "null_instrument", "none", ""]
    
    for vid_id, timeline in video_timelines.items():
        spans_by_inst = extract_action_spans(timeline)
        qa_pairs = []
        
        # 1. Build a chronological list of ALL valid events across ALL instruments
        valid_events = []
        for inst, actions in spans_by_inst.items():
            if inst in exclude_list:
                continue
                
            for action in actions:
                if action["verb"] not in exclude_list and action["target"] not in exclude_list:
                    # Append the instrument name into the dictionary for easy access
                    event_data = dict(action)
                    event_data["inst"] = inst
                    valid_events.append(event_data)
        
        # Sort events by their starting frame to establish a global timeline sequence
        valid_events.sort(key=lambda x: x["start_frame"])

        # Frame lookup map for Concurrency questions
        frame_lookup = { data["frame"]: data["state"] for data in timeline }
        
        # 2. Generate Templates
        for i, current_event in enumerate(valid_events):
            inst = current_event["inst"]
            verb = current_event["verb"]
            target = current_event["target"]
            start = current_event["start_frame"]
            anchor_start = current_event["start_frame"]
            anchor_end = current_event["end_frame"]

            # --- Intra-instrument Sequence (Same tool, next action) ---
            # Find the sub-list of actions for just this one instrument
            inst_sublist = [e for e in valid_events if e["inst"] == inst]
            inst_idx = inst_sublist.index(current_event)
            
            if inst_idx + 1 < len(inst_sublist):
                next_inst_event = inst_sublist[inst_idx + 1]
                next_verb = next_inst_event["verb"]
                next_target = next_inst_event["target"]
                next_start = next_inst_event["start_frame"]
                next_end = next_inst_event["end_frame"]
                
                qa_pairs.append((f"What action did the {inst} perform immediately after {verb}ing the {target}?", f"{next_verb}", '0', anchor_start, anchor_end, next_start, next_end))
                qa_pairs.append((f"What tissue did the {inst} interact with immediately after {verb}ing the {target}?", f"{next_target}", '0', anchor_start, anchor_end, next_start, next_end))

            # --- Inter-instrument Sequence (Different tool, next action overall) ---
            if i + 1 < len(valid_events):
                next_event = valid_events[i + 1]
                next_inst = next_event["inst"]
                next_verb = next_event["verb"]
                next_target = next_event["target"]
                
                # Check if the very next action in the video belongs to a DIFFERENT instrument
                if next_inst != inst:
                    # Template: What instrument 2 does after instrument 1
                    qa_pairs.append((f"What action did the {next_inst} perform immediately after the {inst} began {verb}ing the {target}?", f"{next_verb}", '1', anchor_start, anchor_end, next_event["start_frame"], next_event["end_frame"]))
                    
                    # Template: What instrument 1 did before instrument 2
                    qa_pairs.append((f"What action did the {inst} perform immediately before the {next_inst} began {next_verb}ing the {next_target}?", f"{verb}", '1', anchor_start, anchor_end, next_event["start_frame"], next_event["end_frame"]))

            # --- Concurrency (Simultaneous actions) ---
            frame_state = frame_lookup[start]
            for other_inst, (other_verb, other_target) in frame_state.items():
                if other_inst == inst or other_verb in exclude_list or other_target in exclude_list:
                    continue
                    
                other_span = find_span_at(spans_by_inst[other_inst], start) or {"start_frame": start, "end_frame": start}
                other_start = other_span["start_frame"]
                other_end = other_span["end_frame"]
                
                qa_pairs.append((f"What action was the {other_inst} performing while the {inst} was {verb}ing the {target}?", f"{other_verb}", '2', anchor_start, anchor_end, other_start, other_end))
                qa_pairs.append((f"What tissue was the {other_inst} {other_verb}ing while the {inst} was {verb}ing the {target}?", f"{other_target}", '2', anchor_start, anchor_end, other_start, other_end))
                qa_pairs.append((f"Which instrument was {other_verb}ing the {other_target} while the {inst} was {verb}ing the {target}?", f"{other_inst}", '2', anchor_start, anchor_end, other_start, other_end))

        # Deduplicate the pairs (concurrency checks can sometimes generate the same question twice from inverse perspectives)
        all_qa_pairs[vid_id] = list(set(qa_pairs))
        
    return all_qa_pairs

# ==========================================
# Test Execution Environment
# ==========================================
if __name__ == "__main__":
    input_dir = "tmp_json_single_word_input"
    os.makedirs(input_dir, exist_ok=True)

    def make_json(triplets):
        return json.dumps({"info": {"triplet": triplets}})

    # Mock Sequence:
    # Frame 0: Grasper grasps gallbladder
    # Frame 1: Grasper grasps gallbladder AND Scissors cut cystic_plate
    # Frame 2: Grasper idle AND Scissors dissect cystic_plate
    with open(os.path.join(input_dir, "VID01_0.json"), "w") as f: f.write(make_json(["grasper,grasp,gallbladder"]))
    with open(os.path.join(input_dir, "VID01_1.json"), "w") as f: f.write(make_json(["grasper,grasp,gallbladder", "scissors,cut,cystic_plate"]))
    with open(os.path.join(input_dir, "VID01_2.json"), "w") as f: f.write(make_json(["scissors,dissect,cystic_plate"]))

    timelines = parse_json_sequence(input_dir)
    print(timelines)
    qas = generate_single_word_qa(timelines)

    for vid_id, qa_list in qas.items():
        print(f"=== SINGLE-WORD TEMPORAL QA FOR {vid_id} ===")
        for q, a, t, a_start, a_end, r_start, r_end in qa_list:
            print(f"Q: {q}\nA: {a}\nType: {t} | action frames: {a_start}-{a_end} | related frames: {r_start}-{r_end}\n")

    ## WARNING ## The following lines remove files!!!!!!
    import shutil
    shutil.rmtree(input_dir)

defaultdict(<class 'list'>, {'VID01': [{'frame': 0, 'state': {'grasper': ('grasp', 'gallbladder')}}, {'frame': 1, 'state': {'grasper': ('grasp', 'gallbladder'), 'scissors': ('cut', 'cystic_plate')}}, {'frame': 2, 'state': {'scissors': ('dissect', 'cystic_plate')}}]})
=== SINGLE-WORD TEMPORAL QA FOR VID01 ===
Q: What action did the grasper perform immediately before the scissors began cuting the cystic_plate?
A: grasp
Type: 1 | action frames: 0-1 | related frames: 1-1

Q: Which instrument was grasping the gallbladder while the scissors was cuting the cystic_plate?
A: grasper
Type: 2 | action frames: 1-1 | related frames: 0-1

Q: What action did the scissors perform immediately after the grasper began grasping the gallbladder?
A: cut
Type: 1 | action frames: 0-1 | related frames: 1-1

Q: What tissue did the scissors interact with immediately after cuting the cystic_plate?
A: cystic_plate
Type: 0 | action frames: 1-1 | related frames: 2-2

Q: What action did the scissors perform immediate

In [2]:
input_dir = "SSGVQA/scene_graph"
timelines = parse_json_sequence(input_dir)
qas = generate_single_word_qa(timelines)
qas

{'VID23': [('What action did the grasper perform immediately before the clipper began cliping the cystic_artery?',
   'retract',
   '1',
   670,
   670,
   672,
   685),
  ('What action was the grasper performing while the scissors was cuting the cystic_artery?',
   'retract',
   '2',
   694,
   702,
   693,
   695),
  ('What action did the hook perform immediately after the grasper began retracting the gallbladder?',
   'dissect',
   '1',
   1013,
   1025,
   1014,
   1023),
  ('What tissue was the hook dissecting while the grasper was retracting the omentum?',
   'omentum',
   '2',
   123,
   130,
   82,
   186),
  ('What tissue did the grasper interact with immediately after retracting the omentum?',
   'omentum',
   '0',
   123,
   130,
   132,
   163),
  ('What action did the hook perform immediately before the grasper began retracting the omentum?',
   'coagulate',
   '1',
   69,
   81,
   79,
   86),
  ('What action did the grasper perform immediately after grasping the specimen

In [3]:
output_dir = './ssgqa_qa_temporal/'
os.makedirs(output_dir, exist_ok=True)

# Question|Answer|Question Type|Start frame of action|End frame of action|Start frame of related action|End frame of related action
for knm in qas.keys():
    with open(output_dir+knm+'.txt','w') as f:
        for ll in qas[knm]:
            f.write('|'.join(str(v) for v in ll)+'\n')

In [4]:
fldn = './ssgqa_qa_temporal/'
in_fls = glob.glob(fldn+'/*.txt')
in_fls

['./ssgqa_qa_temporal/VID35.txt',
 './ssgqa_qa_temporal/VID42.txt',
 './ssgqa_qa_temporal/VID25.txt',
 './ssgqa_qa_temporal/VID12.txt',
 './ssgqa_qa_temporal/VID52.txt',
 './ssgqa_qa_temporal/VID02.txt',
 './ssgqa_qa_temporal/VID70.txt',
 './ssgqa_qa_temporal/VID36.txt',
 './ssgqa_qa_temporal/VID29.txt',
 './ssgqa_qa_temporal/VID110.txt',
 './ssgqa_qa_temporal/VID68.txt',
 './ssgqa_qa_temporal/VID47.txt',
 './ssgqa_qa_temporal/VID08.txt',
 './ssgqa_qa_temporal/VID18.txt',
 './ssgqa_qa_temporal/VID14.txt',
 './ssgqa_qa_temporal/VID49.txt',
 './ssgqa_qa_temporal/VID66.txt',
 './ssgqa_qa_temporal/VID10.txt',
 './ssgqa_qa_temporal/VID56.txt',
 './ssgqa_qa_temporal/VID92.txt',
 './ssgqa_qa_temporal/VID50.txt',
 './ssgqa_qa_temporal/VID96.txt',
 './ssgqa_qa_temporal/VID73.txt',
 './ssgqa_qa_temporal/VID103.txt',
 './ssgqa_qa_temporal/VID05.txt',
 './ssgqa_qa_temporal/VID22.txt',
 './ssgqa_qa_temporal/VID15.txt',
 './ssgqa_qa_temporal/VID111.txt',
 './ssgqa_qa_temporal/VID01.txt',
 './ssgqa_q